# L10 · Prompt chaining: pasos fijos con una compuerta

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

El primer patrón de flujo agéntico es el más simple: **encadenar prompts**. Una tarea se divide en pasos fijos y la salida de cada uno es la entrada del siguiente. Entre pasos podemos poner una **compuerta** (*gate*): una comprobación en código que decide si seguir o parar.

El caso: un mensaje de un cliente de **ACME** sobre su crédito. Primero lo **clasificamos**; si no es un tema de crédito, paramos. Si lo es, **extraemos** los datos y **redactamos** una respuesta con la política correspondiente.

```mermaid
flowchart LR
    S([START]) --> C["classify<br/>Claude · salida estructurada"]
    C -->|gate: ¿es de crédito?| X["extract<br/>Claude · datos del ticket"]
    C -->|no| O["out_of_scope<br/>sin modelo"]
    X --> D["draft<br/>Claude · respuesta con la política"]
    D --> E([END])
    O --> E
```

## Objetivos

1. Dividir una tarea en pasos fijos dentro de un grafo.
2. Poner una compuerta en código entre dos pasos.
3. Pasar la salida estructurada de un paso como entrada del siguiente.
4. Medir cuántas llamadas hace cada camino.

El recorrido realiza **4 llamadas a Claude Haiku**: 3 para el mensaje de crédito y 1 para el que no lo es.

## 0. Preparación

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=500,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)

In [ ]:
from pathlib import Path

from pypdf import PdfReader


def policy_text(name):
    """El texto completo de una política: cada una tiene dos páginas y cabe entera en el prompt."""
    path = Path("../datasets") / name
    return "\n\n".join(f"[{name}, p. {n}]\n{page.extract_text()}" for n, page in enumerate(PdfReader(path).pages, 1))

In [ ]:
from IPython.display import Markdown, display


def show(graph):
    """Dibuja el grafo como Mermaid: JupyterLab lo renderiza sin conexión a internet."""
    display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

## 1. El estado y los esquemas

Cada paso deja su resultado en un campo del estado. Las salidas del modelo son estructuradas, como en L11, para que la compuerta pueda leerlas en código.

In [ ]:
from typing import Literal, TypedDict

from pydantic import BaseModel, Field


class Classification(BaseModel):
    """Clasificación de un mensaje de cliente."""

    topic: Literal["credito", "otro"] = Field(description="credito si habla de un crédito, su mora, desembolso o pagos.")


class TicketData(BaseModel):
    """Datos del ticket para responder."""

    customer: str | None = Field(description="Código del cliente C-1234; None si no aparece.")
    question: str = Field(description="La pregunta concreta del cliente, en una frase.")
    policy: Literal["acme_credit_policy.pdf", "acme_disbursement_policy.pdf", "acme_fees_guide.pdf", "acme_collections_policy.pdf"] = Field(
        description="La política que responde la pregunta: crédito, desembolsos, costos y pagos, o cobranza."
    )


class ChainState(TypedDict, total=False):
    message: str
    topic: str
    data: TicketData
    reply: str

## 2. Los pasos

Cada nodo es una llamada con un solo propósito. `out_of_scope` no llama al modelo: es una respuesta fija.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def classify(state: ChainState) -> dict:
    result = model.with_structured_output(Classification).invoke([HumanMessage(state["message"])])
    return {"topic": result.topic}


def extract(state: ChainState) -> dict:
    return {"data": model.with_structured_output(TicketData).invoke([HumanMessage(state["message"])])}


def draft(state: ChainState) -> dict:
    data = state["data"]
    prompt = [
        SystemMessage(
            "Eres el asistente de soporte de ACME. Responde en español, en máximo cuatro frases, solo con la política "
            "que recibes, y cita cada dato con [archivo, p. N]. No prometas aprobaciones."
        ),
        HumanMessage(f"Política:\n{policy_text(data.policy)}\n\nCliente: {data.customer}\nPregunta: {data.question}"),
    ]
    return {"reply": model.invoke(prompt).text}


def out_of_scope(state: ChainState) -> dict:
    return {"reply": "Este canal atiende temas de créditos de ACME. Para otros temas, escribe a soporte general."}

## 3. La compuerta

La compuerta es una función que **lee** el estado y devuelve el nombre del siguiente nodo. No modifica nada. Con `add_conditional_edges`, LangGraph llama a esa función después de `classify` y sigue por el camino que devuelve. Es el mismo mecanismo del *routing* de L14, aquí con una sola salida alternativa.

In [ ]:
from langgraph.graph import END, START, StateGraph


def gate(state: ChainState) -> Literal["extract", "out_of_scope"]:
    return "extract" if state["topic"] == "credito" else "out_of_scope"


builder = StateGraph(ChainState)
builder.add_node("classify", classify)
builder.add_node("extract", extract)
builder.add_node("draft", draft)
builder.add_node("out_of_scope", out_of_scope)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", gate)
builder.add_edge("extract", "draft")
builder.add_edge("draft", END)
builder.add_edge("out_of_scope", END)

chain = builder.compile()
show(chain)

## 4. Dos mensajes, dos caminos

El primero es de crédito y recorre los tres pasos (**3 llamadas**); el segundo se detiene en la compuerta (**1 llamada**). `stream_mode="updates"` muestra qué escribió cada nodo.

In [ ]:
for message in (
    "Soy C-3307, tengo 75 días de mora. ¿Me pueden reportar a centrales de riesgo?",
    "¿Ustedes venden seguros de carro?",
):
    print("=====", message)
    for step in chain.stream({"message": message}, stream_mode="updates"):
        node, update = next(iter(step.items()))
        print(f"[{node}]", update.get("topic") or update.get("data") or update.get("reply"))

Revisa el primer camino: ¿`extract` eligió la política de cobranza? ¿La respuesta cita la página donde dice que el reporte es desde los 61 días? Si la clasificación falla, la compuerta corta el flujo antes de gastar dos llamadas más: ese es el valor de poner la comprobación **entre** pasos.

## Práctica breve

1. Añade una segunda compuerta después de `extract`: si `customer` es `None`, responde pidiendo el código del cliente sin llamar a `draft`.
2. Cuenta las llamadas de cada camino con `stream` y compáralas con un solo prompt que haga todo. ¿Qué ganas y qué pierdes?
3. Cambia `draft` para que use el retriever de L4 en lugar del texto completo de la política.

## Referencias

- **Anthropic**, [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents): *prompt chaining*.
- [Workflows y agentes en LangGraph](https://docs.langchain.com/oss/python/langgraph/workflows-agents)